# Notebook 1: PATSTAT High-Recall Family Baseline Extraction
Extracts candidate drone-related **innovation patents** (`ipr_type = 'PI'`, excluding utility models) from PATSTAT `PROD`, normalizes whitespace in CPC/IPC codes, filters out PATSTAT dummy records and textless entries, deduplicates to **one representative record per `docdb_family_id`** (prioritizing English-language filings for accurate downstream embeddings while preserving the earliest family filing year), and saves the baseline corpus to Parquet.

In [1]:
import os
from datetime import date

import pandas as pd
from sqlalchemy import func, or_, and_, not_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
# Connect to PATSTAT PROD environment
patstat = PatstatClient(env="PROD")
db = patstat.orm()

In [3]:
# Time window constants (excludes 2023+ due to 18-month publication & PATSTAT ingestion lag)
MIN_FILING_DATE = date(2000, 1, 1)
MAX_FILING_DATE = date(2023, 1, 1)
LAST_COMPLETE_YEAR = 2022
YEAR_MAX = 2025

# Multi-tier taxonomy and exclusion lists

# Primary Anchor Keywords (High Precision: Directly define flying UAV platforms)
DRONE_ANCHOR_KEYWORDS = [
    "%unmanned aerial vehicle%",
    "%unmanned aircraft%",
    "%remotely piloted aircraft%",
    "%RPAS%",
    "%UAV%",
]

# Secondary Functional Keywords (Dual-Use: Require co-occurrence with Core CPC/IPC or Anchor Keywords)
DRONE_SECONDARY_KEYWORDS = [
    "%drone%",
    "%quadcopter%",
    "%multirotor%",
    "%hexacopter%",
]

# Exclusion Keywords (Filter out non-flying autonomous/robotic domains)
DRONE_EXCLUSION_KEYWORDS = [
    "%toy%",
    "%underwater%",
    "%submersible%",
    "%marine%",
]

DRONE_CORE_CPC_CODES = [
    "B64U%",        # Primary CPC class dedicated to UAVs
    "B64C39/02%",   # Legacy CPC class for remote-controlled aircraft
]

DRONE_SECONDARY_CPC_CODES = [
    "G08G5/00%",    # Aeronautical traffic control & UTM
    "G05D1/00%",    # Control of position / course / flight control
]

DRONE_CORE_IPC_CODES = [
    "B64U%",
    "B64C39/02%",
]

DRONE_SECONDARY_IPC_CODES = [
    "G08G5/00%",
    "G05D1/00%",
]

In [4]:
# Helper functions
def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: blank-stripped CPC/IPC symbol starts with one of the target prefixes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.rstrip("%").replace(" ", "") + "%") for code in codes))

# PCT international application indicator (appln_auth = 'WO' and appln_kind starting with 'W')
IS_PCT_APPLN = and_(TLS201_APPLN.appln_auth == "WO", TLS201_APPLN.appln_kind.like("W%"))

def shade_truncated(ax):
    """Shade incomplete (truncated) years on a time axis."""
    ax.axvspan(LAST_COMPLETE_YEAR + 0.5, YEAR_MAX + 0.5, color="grey", alpha=0.15, lw=0,
               label="incomplete years")

def complete_years(df, year_col="earliest_filing_year"):
    """Filter DataFrame to complete filing years only."""
    return df[df[year_col] <= LAST_COMPLETE_YEAR]

In [5]:
# Build Atomic Filter Conditions (stripping internal blanks from PATSTAT CPC/IPC symbols)
core_cpc_cond = symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_CORE_CPC_CODES)
core_ipc_cond = symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_CORE_IPC_CODES)

sec_cpc_cond = symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, DRONE_SECONDARY_CPC_CODES)
sec_ipc_cond = symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, DRONE_SECONDARY_IPC_CODES)

title_anchor_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]
abstr_anchor_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_ANCHOR_KEYWORDS]

title_sec_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]
abstr_sec_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_SECONDARY_KEYWORDS]

title_excl_f = [TLS202_APPLN_TITLE.appln_title.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]
abstr_excl_f = [TLS203_APPLN_ABSTR.appln_abstract.ilike(k) for k in DRONE_EXCLUSION_KEYWORDS]

In [6]:
# Restructured Boolean Logic Clauses
standalone_core = or_(
    core_cpc_cond,
    core_ipc_cond,
    *title_anchor_f,
    *abstr_anchor_f
)

# Secondary codes MUST co-occur with explicit Anchor terms
secondary_code_match = and_(
    or_(sec_cpc_cond, sec_ipc_cond),
    or_(*title_anchor_f, *abstr_anchor_f)
)

# Secondary keywords MUST co-occur with Core Codes
secondary_kw_match = and_(
    or_(*title_sec_f, *abstr_sec_f),
    or_(core_cpc_cond, core_ipc_cond)
)

# Master Conditions
inclusion_clause = or_(
    standalone_core,
    secondary_code_match,
    secondary_kw_match
)

exclusion_clause = not_(or_(*title_excl_f, *abstr_excl_f))

In [7]:
query = (
    db.query(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS201_APPLN.ipr_type,
        IS_PCT_APPLN.label("is_pct"),
        TLS202_APPLN_TITLE.appln_title_lg,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract_lg,
        TLS203_APPLN_ABSTR.appln_abstract,
        func.string_agg(func.distinct(TLS206_PERSON.doc_std_name), ', ').label("applicants"),
        func.string_agg(func.distinct(TLS224_APPLN_CPC.cpc_class_symbol), ', ').label("cpc_codes"),
        func.string_agg(func.distinct(TLS209_APPLN_IPC.ipc_class_symbol), ', ').label("ipc_codes"),
    )
    .join(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
    .outerjoin(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
    .outerjoin(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .filter(
        and_(
            inclusion_clause,
            exclusion_clause,
            TLS201_APPLN.ipr_type == "PI",                        # Innovation patents only (exclude utility models 'UM' & designs 'DP')
            TLS201_APPLN.appln_id > 0,                            # Exclude PATSTAT dummy appln_id = 0
            TLS201_APPLN.appln_filing_date != date(9999, 12, 31), # Exclude PATSTAT dummy date
            TLS201_APPLN.appln_filing_date >= MIN_FILING_DATE,
            TLS201_APPLN.appln_filing_date < MAX_FILING_DATE
        )
    )
    .group_by(
        TLS201_APPLN.appln_id,
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_nr,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.appln_kind,
        TLS201_APPLN.ipr_type,
        TLS202_APPLN_TITLE.appln_title_lg,
        TLS202_APPLN_TITLE.appln_title,
        TLS203_APPLN_ABSTR.appln_abstract_lg,
        TLS203_APPLN_ABSTR.appln_abstract
    )
)

### Execute Query on PATSTAT PROD
**Warning:** Running `query.all()` queries the full production database and takes several minutes. Only run when regenerating the raw baseline.

In [8]:
results = query.all()
df = pd.DataFrame(results)
print(f"Total matching innovation patent rows retrieved: {len(df):,}")

Total matching innovation patent rows retrieved: 128,597


### Baseline Data Quality Diagnostics & Cleaning
Checks for utility model leakage, PATSTAT dummy IDs/dates, blank CPC/IPC codes, and missing titles/abstracts before deduplicating into DOCDB patent families.

In [9]:
# 1. Check IPR type & utility model exclusion
print("IPR Type distribution:")
print(df['ipr_type'].value_counts(dropna=False))

# 2. Check PATSTAT dummy appln_id, docdb_family_id, and filing dates
df['filing_date_dt'] = pd.to_datetime(df['appln_filing_date'], errors='coerce')
df['filing_year'] = df['filing_date_dt'].dt.year

dummy_appln = (df['appln_id'] <= 0) | (df['appln_id'].isna())
dummy_fam = (df['docdb_family_id'] <= 0) | (df['docdb_family_id'].isna())
dummy_date = df['filing_year'].isna() | (df['filing_year'] >= 9999)

print(f"\nDummy/invalid appln_id count: {dummy_appln.sum()}")
print(f"Dummy/invalid docdb_family_id count: {dummy_fam.sum()}")
print(f"Dummy/invalid filing_date count: {dummy_date.sum()}")

df = df[~(dummy_appln | dummy_fam | dummy_date)].copy()

# 3. Check blank/missing CPC and IPC codes
cpc_blank = df['cpc_codes'].fillna('').str.strip() == ''
ipc_blank = df['ipc_codes'].fillna('').str.strip() == ''
print(f"\nBlank/missing CPC codes: {cpc_blank.sum():,} ({cpc_blank.mean():.2%})")
print(f"Blank/missing IPC codes: {ipc_blank.sum():,} ({ipc_blank.mean():.2%})")
print(f"Blank/missing BOTH CPC & IPC: {(cpc_blank & ipc_blank).sum():,}")

# 4. Check missing/blank Title AND Abstract
title_blank = df['appln_title'].fillna('').str.strip() == ''
abstr_blank = df['appln_abstract'].fillna('').str.strip() == ''
both_blank = title_blank & abstr_blank

print(f"\nBlank/missing Title: {title_blank.sum():,} ({title_blank.mean():.2%})")
print(f"Blank/missing Abstract: {abstr_blank.sum():,} ({abstr_blank.mean():.2%})")
print(f"Blank/missing BOTH Title & Abstract: {both_blank.sum():,}")

# Drop rows where BOTH title and abstract are missing
df = df[~both_blank].copy().reset_index(drop=True)
print(f"\nValid rows remaining before family deduplication: {len(df):,}")

IPR Type distribution:
ipr_type
PI    128597
Name: count, dtype: int64

Dummy/invalid appln_id count: 0
Dummy/invalid docdb_family_id count: 0
Dummy/invalid filing_date count: 0

Blank/missing CPC codes: 12,720 (9.89%)
Blank/missing IPC codes: 369 (0.29%)
Blank/missing BOTH CPC & IPC: 151

Blank/missing Title: 0 (0.00%)
Blank/missing Abstract: 0 (0.00%)
Blank/missing BOTH Title & Abstract: 0

Valid rows remaining before family deduplication: 128,597


### Language-Aware DOCDB Patent Family Deduplication (`docdb_family_id`)
Aggregates family-level metadata (`family_size`, `family_auths`, `has_pct`, `earliest_filing_date`, `earliest_filing_year`) across **all** family members first, then selects one representative record per `docdb_family_id` using the following priority chain:
1. **English language (`is_english = True`):** Keeps an English title/abstract if any family member has one, falling back seamlessly to the original language when no English member exists.
2. **Abstract availability (`has_abstract = True`):** Ensures downstream embeddings have both title and abstract text.
3. **Major English-examining offices (`US`, `EP`, `WO`, `GB`, `CA`, `AU`):** Prefers native English specifications over machine-translated domestic abstracts when multiple English versions exist.
4. **Earliest filing date & lowest `appln_id`:** Breaks remaining ties deterministically.

In [10]:
# 1. Compute family-level summary attributes across ALL members before deduplicating
family_stats = (
    df.groupby('docdb_family_id', as_index=False)
    .agg(
        family_size=('appln_id', 'nunique'),
        family_auths=('appln_auth', lambda s: ', '.join(sorted(set(s.dropna().str.strip())))),
        has_pct=('is_pct', 'any'),
        earliest_filing_date=('filing_date_dt', 'min'),
        earliest_filing_year=('filing_year', 'min'),
    )
)

# 2. Flag English availability, abstract presence, and preferred English patent offices
title_en = df['appln_title_lg'].fillna('').str.strip().str.lower() == 'en'
abstr_en = df['appln_abstract_lg'].fillna('').str.strip().str.lower() == 'en'

df['is_english'] = title_en | abstr_en
df['has_abstract'] = df['appln_abstract'].fillna('').str.strip() != ''

en_office_rank = {'US': 1, 'EP': 2, 'WO': 3, 'GB': 4, 'CA': 5, 'AU': 6}
df['auth_priority'] = df['appln_auth'].fillna('').str.strip().map(en_office_rank).fillna(9)

# 3. Sort so English-language representatives come first, falling back to earliest filing date
df_sorted = df.sort_values(
    by=[
        'docdb_family_id',
        'is_english',
        'has_abstract',
        'auth_priority',
        'filing_year',
        'filing_date_dt',
        'appln_id',
    ],
    ascending=[True, False, False, True, True, True, True]
)

# 4. Keep top representative per DOCDB patent family and merge family-level stats
df_families = (
    df_sorted.drop_duplicates(subset=['docdb_family_id'], keep='first')
    .drop(columns=['has_abstract', 'auth_priority', 'filing_date_dt'])
    .merge(family_stats, on='docdb_family_id', how='left')
    .reset_index(drop=True)
)

print(f"Total Applications (appln_id): {df['appln_id'].nunique():,}")
print(f"Total Deduplicated Patent Families (docdb_family_id): {len(df_families):,}")
print(f"Families with English text selected: {df_families['is_english'].sum():,} ({df_families['is_english'].mean():.2%})")

print("\nLanguage distribution of selected family representatives (Title language):")
print(df_families['appln_title_lg'].fillna('missing').value_counts().head(10))

print("\nEarliest Filing Year distribution (DOCDB Families):")
print(df_families['earliest_filing_year'].value_counts().sort_index())

df_families.head()

Total Applications (appln_id): 128,597
Total Deduplicated Patent Families (docdb_family_id): 95,426
Families with English text selected: 89,826 (94.13%)

Language distribution of selected family representatives (Title language):
appln_title_lg
en    89795
es     1864
pt     1660
no      953
de      761
fr      221
tr       55
nl       35
ko       32
ja       31
Name: count, dtype: int64

Earliest Filing Year distribution (DOCDB Families):
earliest_filing_year
2000      599
2001      569
2002      519
2003      469
2004      589
2005      658
2006      652
2007      742
2008      796
2009      719
2010      753
2011      914
2012     1110
2013     1496
2014     2207
2015     4136
2016     7842
2017    10579
2018    11343
2019    10845
2020    12044
2021    12639
2022    13206
Name: count, dtype: int64


,appln_id,docdb_family_id,appln_auth,appln_nr,appln_filing_date,appln_kind,ipr_type,is_pct,appln_title_lg,appln_title,...,applicants,cpc_codes,ipc_codes,filing_year,is_english,family_size,family_auths,has_pct,earliest_filing_date,earliest_filing_year
0,341742139,3460785,AR,P000100142,2000-01-13,A,PI,False,es,DISPOSICIoN AUToNOMA DE TRANSPORTE Y VEHíCULO ...,...,SERRANO JORGE,"Y02T 30/00, B61B 13/00, B66B 9/00, B66B 1...","B66B 1/14, B66B 1/24, B66B 11/04, B61B 1...",2000,False,1,AR,False,2000-01-13,2000
1,4781163,3491672,CA,2410377,2001-06-28,A,PI,False,en,MOTOR VEHICLE DOOR WITH A PASSENGER PROTECTION...,...,"STEYR DAIMLER PUCH AG, BINDAR GERNOT, DIETZ WO...",B60J 5/0443,B60J 5/04,2001,True,1,CA,False,2001-06-28,2001
2,17637526,3519941,ES,00120245,2000-09-27,T,PI,False,es,AGLUTINANTES DILUIBLES CON AGUA PARA BARNICES ...,...,"PETRITSCH GERLINDE, SCHAFHEUTLE MARKUS DR, BUR...","C08G 18/0823, C08G 18/6659, C09D 175/06","C09D 161/26, C09D 5/00, C09D 175/06, C08G 1...",2000,False,1,ES,False,2000-09-27,2000
3,12969091,3819479,DE,10103940,2001-01-30,A,PI,False,de,Vorrichtung zum Einspeisen von Gas in einen Ke...,...,"DUNNE MARTIN JOSEPH, TECH RESOURCES PTY LTD","C21C 5/4606, F27B 3/22, F27D 3/16, C21B ...","C22B 9/05, C21C 7/072, F27D 3/16, C21B ...",2001,False,1,DE,False,2001-01-30,2001
4,3481792,3826739,BR,0200055,2002-01-10,A,PI,False,pt,equipamento de esmagamento de materiais.,...,BOND NEVILLE JAMES,"B02C 4/30, B02C 4/08","B02C 17/14, B02C 4/30, B02C 4/08",2002,False,1,BR,False,2002-01-10,2002


In [12]:
# Save family-deduplicated baseline corpus to Parquet
baseline_path = "flying_drones_baseline_prod_final.parquet"
df_families.to_parquet(baseline_path, index=False)

size_mb = os.path.getsize(baseline_path) / (1024 * 1024)
print(f"Successfully saved '{baseline_path}': {len(df_families):,} patent families ({size_mb:.2f} MB)")

Successfully saved 'flying_drones_baseline_prod_final.parquet': 95,426 patent families (56.69 MB)
